In [ ]:
"""MARK: Colab Ollama Server — GPU + Cloudflare tunnel."""

MODEL = "qwen3.5:9b" # @param {type:"string"}
PORT = "11434"

import os, re, time, subprocess as sp

def run(cmd, check=True):
    return sp.run(cmd, shell=True, text=True, check=check)

def out(cmd):
    return sp.check_output(cmd, shell=True, text=True).strip()

gpu = out("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader")
assert gpu, "No GPU"
print("GPU:", gpu)

run("apt-get update -qq")
run("apt-get install -y -qq zstd curl wget")

run("curl -fsSL https://ollama.com/download/ollama-linux-amd64.tar.zst | tar --zstd -x -C /usr")
run("wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared")
run("chmod +x /usr/local/bin/cloudflared")
print("Installed")

run("pkill -f 'ollama serve' || true", check=False)
sp.Popen(
    ["ollama", "serve"],
    env={
        **os.environ,
        "OLLAMA_HOST": f"0.0.0.0:{PORT}",
        "OLLAMA_KEEP_ALIVE": "-1",
        "OLLAMA_ORIGINS": "*",
        "CUDA_VISIBLE_DEVICES": "0",
    },
    stdin=sp.DEVNULL,
    stdout=sp.DEVNULL,
    stderr=sp.DEVNULL,
)

for _ in range(60):
    if sp.run(f"curl -fsS http://127.0.0.1:{PORT}/api/tags >/dev/null", shell=True).returncode == 0:
        break
    time.sleep(1)
else:
    raise RuntimeError("Ollama failed to start")

print("Ollama started")

run("pip install -q ollama")
from ollama import Client

client = Client(host=f"http://127.0.0.1:{PORT}")

models = [m.model for m in client.list().models]
if MODEL not in models:
    print("Pulling", MODEL)
    client.pull(MODEL)

print("Warmup:", end=" ", flush=True)
for chunk in client.chat(
    model=MODEL,
    messages=[{"role": "user", "content": "Say hi briefly."}],
    stream=True,
    think=False,
    keep_alive=-1,
):
    print(chunk.message.content, end="", flush=True)
print()

print(out("ollama ps"))

tunnel = sp.Popen(
    ["cloudflared", "tunnel", "--url", f"http://localhost:{PORT}"],
    stdout=sp.PIPE,
    stderr=sp.STDOUT,
    text=True,
)

for line in tunnel.stdout:
    m = re.search(r"(https://[^\s]+\.trycloudflare\.com)", line)
    if m:
        print("\n" + "=" * 50)
        print(m.group(1))
        print("=" * 50 + "\n")
        break

print("Running")
try:
    while True:
        time.sleep(60)
except KeyboardInterrupt:
    print("Stopped")